In [ ]:
print("start!!!!!!!!!!!")

start!!!!!!!!!!!


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import zscore





In [ ]:
df= pd.read_csv("DATA/DataCoSupplyChainDataset.csv",  encoding="latin-1")

df.shape

(180519, 53)

In [ ]:
df.head(3)

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class


In [ ]:
print("Dataset column: ", df.columns.tolist())
print("Data types: ", df.dtypes)
print("Missing values: ", df.isnull().sum())
print("Summary statistics: ", df.describe())

Dataset column:  ['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Password', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Description', 'Product Image', 'Product Name', 'Product Price', 'Product Status', 'shippi

In [ ]:
columns = [

    "Shipping Mode",
    "Delivery Status",
    "Late_delivery_risk",
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Order Profit Per Order",
    "Sales",
    "Order Item Discount Rate",
    "Customer Segment",
    "Market",
    "Category Name"
]

dataset = df[columns].copy()

path= "DATA/finalDatas.csv"
dataset.to_csv(path, index=False)

dataset.shape

(180519, 11)

In [ ]:
dataset.head(4)

,Shipping Mode,Delivery Status,Late_delivery_risk,Days for shipping (real),Days for shipment (scheduled),Order Profit Per Order,Sales,Order Item Discount Rate,Customer Segment,Market,Category Name
0,Standard Class,Advance shipping,0,3,4,91.250000,327.75,0.04,Consumer,Pacific Asia,Sporting Goods
1,Standard Class,Late delivery,1,5,4,-249.089996,327.75,0.05,Consumer,Pacific Asia,Sporting Goods
2,Standard Class,Shipping on time,0,4,4,-247.779999,327.75,0.06,Consumer,Pacific Asia,Sporting Goods
3,Standard Class,Advance shipping,0,3,4,22.860001,327.75,0.07,Home Office,Pacific Asia,Sporting Goods


In [ ]:
print("Dataset column: ", dataset.columns.tolist())
print("Data types: ", dataset.dtypes)
print("Missing values: ", dataset.isnull().sum())
print("Summary statistics: ", dataset.describe())

Dataset column:  ['Shipping Mode', 'Delivery Status', 'Late_delivery_risk', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Order Profit Per Order', 'Sales', 'Order Item Discount Rate', 'Customer Segment', 'Market', 'Category Name']
Data types:  Shipping Mode                        str
Delivery Status                      str
Late_delivery_risk                 int64
Days for shipping (real)           int64
Days for shipment (scheduled)      int64
Order Profit Per Order           float64
Sales                            float64
Order Item Discount Rate         float64
Customer Segment                     str
Market                               str
Category Name                        str
dtype: object
Missing values:  Shipping Mode                    0
Delivery Status                  0
Late_delivery_risk               0
Days for shipping (real)         0
Days for shipment (scheduled)    0
Order Profit Per Order           0
Sales                            0
Order Item Di

DATA CLEANING STEPS: Duplicate datas, outliers

In [ ]:

duplicates = dataset.duplicated().sum()
print("Number of duplicate rows: ", duplicates)

Number of duplicate rows:  6364


In [ ]:
if duplicates > 0:
    dataset = dataset.drop_duplicates()
    print("Duplicate rows removed. New shape: ", dataset.shape)

Duplicate rows removed. New shape:  (174155, 11)


In [ ]:
invalid_dimensions_mask = (dataset['Late_delivery_risk'] <= 0) | (dataset['Days for shipping (real)'] <= 0) | (dataset['Days for shipment (scheduled)'] <= 0) | (dataset['Order Profit Per Order'] <= 0) | (dataset['Sales'] <= 0)| (dataset['Order Item Discount Rate'] <= 0)
invalid_count = invalid_dimensions_mask.sum()
print("Number of rows with invalid dimensions: ", invalid_count)

Number of rows with invalid dimensions:  105367


In [ ]:
if invalid_count > 0:
    dataset = dataset[~invalid_dimensions_mask]
    print("Invalid dimension rows removed. New shape: ", dataset.shape)

Invalid dimension rows removed. New shape:  (68788, 11)


In [ ]:
Cols=['Days for shipping (real)',
    'Days for shipment (scheduled)',
    'Order Profit Per Order',
    'Sales',
    'Order Item Discount Rate']
threshold = 2
for c in Cols:
    z_scores= zscore(dataset[c])
    outliers = np.where(np.abs(z_scores) > threshold) 
    
    print(f"Outliers in {c} : {len(outliers[0])} ")


Outliers in Days for shipping (real) : 0 
Outliers in Days for shipment (scheduled) : 0 
Outliers in Order Profit Per Order : 2636 
Outliers in Sales : 1388 
Outliers in Order Item Discount Rate : 4112 


In [ ]:

capping= dataset.copy()

z= zscore(dataset['Order Profit Per Order'])
capping['Order Profit Per Order'] = np.where( z > threshold,
                            dataset['Order Profit Per Order'].mean() + threshold * dataset['Order Profit Per Order'].std(),
                            dataset['Order Profit Per Order'])
capping['Order Profit Per Order'] = np.where(z < -threshold,
                            dataset['Order Profit Per Order'].mean() - threshold * dataset['Order Profit Per Order'].std(),
                            capping['Order Profit Per Order'])



z= zscore(dataset['Sales'])
capping['Sales'] = np.where( z > threshold,
                            dataset['Sales'].mean() + threshold * dataset['Sales'].std(),
                            dataset['Sales'])
capping['Sales'] = np.where(z < -threshold,
                            dataset['Sales'].mean() - threshold * dataset['Sales'].std(),
                            capping['Sales'])


z= zscore(dataset['Order Item Discount Rate'])
capping['Order Item Discount Rate'] = np.where( z > threshold,
                            dataset['Order Item Discount Rate'].mean() + threshold * dataset['Order Item Discount Rate'].std(),
                            dataset['Order Item Discount Rate'])
capping['Order Item Discount Rate'] = np.where(z < -threshold,
                            dataset['Order Item Discount Rate'].mean() - threshold * dataset['Order Item Discount Rate'].std(),
                            capping['Order Item Discount Rate'])


print("Original shape  :", dataset.shape)
print("Capped shape    :", capping.shape)

dataset= capping
dataset.shape


Original shape  : (68788, 11)
Capped shape    : (68788, 11)


(68788, 11)

In [ ]:
cols = ['Shipping Mode', 'Delivery Status', 'Customer Segment', 'Market', 'Category Name']
for col in cols:
    unique_vals = dataset[col].unique()
    print(f"\n[{col}]  has   {len(unique_vals)} unique values")
    for val in sorted(unique_vals):
        count = (df[col] == val).sum()
        print(f"  {repr(val):<35}  {count:>6,}")